<a href="https://colab.research.google.com/github/BeachBall2024/GEO-spa-algo/blob/main/submission.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Workflow and Introduction
Study:
1. Load data and basic functions
- Lea
2. preprocess Flickr data for typos, upper-lower case
3. construct sound wheel
- Julia
4. divide streets into segments (name, then again for a max length)
- Muriel
5. create Buffer around streets
- Guillermo
6. check which points lay inside buffer and match them to sound categories, count

In [1]:
# Import github repo libraries
import os, sys
if not os.path.exists('geo_proxy'):
    print("geo_proxy package not found. If in Colab, please ensure the repository is cloned.")
    # We skip !git clone in local execution for safety
else:
    print("geo_proxy package already in current path.")

geo_proxy package already in current path.


2. Import Python packages

In [2]:
#importing important packages

3. Import basic geometric functions
(Functions are dividet in different classes based on shape)

- Point
- Line
- Polygon

In [3]:
# POINT
class Point():
    def __init__(self, x=None, y=None):
        self.x = x
        self.y = y
    def __repr__(self):
        return f'Point(x={self.x}, y={self.y})'
    def __eq__(self, other):
        if not isinstance(other, Point): return False
        return self.x == other.x and self.y == other.y
    def isEqual(self, other):
        return self.__eq__(other)
    def __hash__(self):
        return hash((self.x, self.y))

In [4]:
# LINE
class Bbox():
    def __init__(self, segment):
        self.min_x = min(segment.start.x, segment.end.x)
        self.max_x = max(segment.start.x, segment.end.x)
        self.min_y = min(segment.start.y, segment.end.y)
        self.max_y = max(segment.start.y, segment.end.y)
    def testOverlap(self, other):
        return not (self.max_x < other.min_x or self.min_x > other.max_x or \
                    self.max_y < other.min_y or self.min_y > other.max_y)

class Segment():
    def __init__(self, p0, p1, sid=None):
        self.start = p0
        self.end = p1
        self.sid = sid
    def __repr__(self):
        return f'Segment with start {self.start} and end {self.end}.'
    def isIdentical(self, other):
        return (self.start.isEqual(other.start) or self.start.isEqual(other.end)) and \
               (self.end.isEqual(other.end) or self.end.isEqual(other.start))
    def intersects(self, other):
        s_bbox = Bbox(self)
        o_bbox = Bbox(other)
        return s_bbox.testOverlap(o_bbox)
    def length(self):
        import math
        return math.sqrt((self.start.x - self.end.x)**2 + (self.start.y - self.end.y)**2)

In [5]:
#POLYGON
# Polygon class for polygons, assumes initial data is in a spatially sorted order
class PointGroup:
    # initialise
    def __init__(self, data=None, xcol=None, ycol=None):
        self.points = []
        self.size = len(data)
        for d in data:
            self.points.append(Point(d[xcol], d[ycol]))

    # representation
    def __repr__(self):
        return f'Polygon PointGroup containing {self.size} points'

    # test if polygon is closed: first and last point should be identical
    def isClosed(self):
        start = self.points[0]
        end = self.points[-1]
        return start == end

    def removeDuplicates(self):
        oldn = len(self.points)
        self.points = list(dict.fromkeys(self.points)) # Get rid of the duplicates
        self.points.append(self.points[0]) # Our polygon must have one duplicate - we put it back now
        n = len(self.points)
        self.size = n   # see how the absence/presence of this line makes changes
        print(f'The old polygon had {oldn} points, now we only have {n}.')

# 4. Data Loading

  We load three datasets:
- **Flickr Images**
- **Sound Wheel**
- **Streets of Switzerland**


In [6]:
# Import datasets
import pandas as pd
import os

# Check path
data_dir = 'data'
if not os.path.exists(data_dir) and os.path.exists('../data'):
    data_dir = '../data'

print(f"Using data directory: {data_dir}")
flickr_points = pd.read_csv(os.path.join(data_dir, 'zurich_sounds.csv'))
noise_data = pd.read_csv(os.path.join(data_dir, 'zurich_noise.csv'))
print(f"Loaded {len(flickr_points)} Flickr points")
print(f"Loaded {len(noise_data)} noise measurements")

Using data directory: data
Loaded 800 Flickr points
Loaded 150 noise measurements


# 5. Preprocessing of Flickr Images



In [7]:
# Preprocessing of Flickr Images
# Normalize tag casing and types
flickr_points['tags'] = flickr_points['tags'].str.lower()
print("Preprocessing complete: all tags converted to lowercase.")

Preprocessing complete: all tags converted to lowercase.


# 6. Division of Streets into Segments

**Task owner: Julia**

The original approach using a GDB file was not feasible (file too large for GitHub, CSV only has centroids).
Instead, we read street geometries directly from the `streets_ZH.shp` shapefile using a **pure-Python SHP parser**
(no external GIS libraries), match street names from the swisstopo CSV via nearest-centroid lookup,
filter to Zürich, and cut each street into segments of ≤ 500 m.

Coordinates are converted from CH1903+/LV95 (EPSG:2056) to WGS84 using the official swisstopo approximate formulas.

**Algorithm:** Walk along each polyline; when accumulated distance exceeds 500 m, interpolate the exact
split point so segments fit together with no gaps. Complexity: O(n) where n = number of vertices.

In [8]:
import csv, math
from collections import defaultdict
from geo_proxy.shp_parser import read_shp
from geo_proxy.street_segmentation import (
    segment_streets, save_segments_csv, lv95_to_wgs84
)

# ── Settings ─────────────────────────────────────────────────────────
SHP_FILE   = 'streets_ZH.shp'
CSV_FILE   = 'amtliches-strassenverzeichnis_ch_2056.csv'
OUT_FILE   = 'data/zurich_street_segments.csv'
MAX_SEG_M  = 500
# ─────────────────────────────────────────────────────────────────────

# Run the full segmentation pipeline
street_segments = segment_streets(
    shp_path=SHP_FILE,
    csv_path=CSV_FILE,
    max_segment_length=MAX_SEG_M,
)

# Save to CSV for later use
save_segments_csv(street_segments, OUT_FILE)

# ── Summary statistics ───────────────────────────────────────────────
named = [s for s in street_segments if s['street_name'] != 'Unknown']
print(f'\nNamed segments: {len(named)} / {len(street_segments)}')

street_lengths = defaultdict(float)
for s in named:
    street_lengths[s['street_name']] += s['length_m']
top5 = sorted(street_lengths.items(), key=lambda x: -x[1])[:5]
print('\nTop 5 longest named streets:')
for name, length in top5:
    print(f'  {name}: {length:.0f} m')

print(f'\nSample segments:')
for s in named[:5]:
    print(f"  {s['street_name']:30s} seg {s['segment_id']}/{s['total_segments']}  "
          f"{s['length_m']:>7.1f}m  ({s['start_lat']}, {s['start_lon']})")

Reading shapefile ...
  26,711 polylines read from shapefile.
  5,049 polylines in Zürich bbox.
Loading street names from CSV ...
  2,573 Zürich street name entries loaded.
Segmenting streets ...
  10,042 segments created.
Saved 10042 segments to data/zurich_street_segments.csv

Named segments: 5559 / 10042

Top 5 longest named streets:
  Rundweg: 64819 m
  Cyklamenweg: 44937 m
  Ligusterstrasse: 37964 m
  Murwiesenweg: 27369 m
  Anna-Häuptli-Weg: 26613 m

Sample segments:
  Häringstrasse                  seg 1/1    464.6m  (47.3767377, 8.5445086)
  Apfelbaumstrasse               seg 1/1    495.1m  (47.4052188, 8.5553136)
  Stelzenstrasse                 seg 1/2    500.0m  (47.4250483, 8.5595703)
  Stelzenstrasse                 seg 2/2     21.5m  (47.4249933, 8.5598283)
  Dreikönigbrücke                seg 1/1     29.3m  (47.3676276, 8.5385393)


# 7. Creation of Buffer

**Task owner: Guillermo**

For each street segment, we construct a **rectangular buffer polygon** at a given distance (50 m default).
This is Algorithm 1 from our pipeline, implemented in `geo_proxy/algorithms.py`.

**Method (Lectures 1, 3):**
1. Compute the direction vector of the segment AB.
2. Rotate 90° to get the perpendicular unit vector.
3. Convert the buffer distance from metres to degrees using Haversine constants at the segment's mid-latitude (Lecture 1).
4. Offset both endpoints by ±perpendicular → four corners of a rectangle.

The resulting `Polygon` object carries a bounding box for fast spatial filtering (Lectures 3–4).

**Complexity:** O(1) per segment — constant number of geometric operations.

We also demonstrate `segments_intersect()` which uses four turn tests (cross products) with a
bounding-box pre-filter (Lecture 3) to detect overlapping buffer regions.

In [9]:
from geo_proxy.primitives import Point, Segment, Polygon
from geo_proxy.algorithms import (
    build_segment_buffer, point_in_polygon, segments_intersect,
    SOUND_CATEGORIES
)

# ── Build Segment objects from our segmented streets ────────────────
# Use only named segments for the analysis
named_segments = [s for s in street_segments if s['street_name'] != 'Unknown']

# Create Segment primitives (lon = x, lat = y in WGS84)
seg_objects = []
seg_names   = []
for s in named_segments:
    p1 = Point(s['start_lon'], s['start_lat'])
    p2 = Point(s['end_lon'],   s['end_lat'])
    seg_objects.append(Segment(p1, p2))
    seg_names.append(s['street_name'])

print(f'Created {len(seg_objects)} Segment objects')

# ── Build buffer polygons (Algorithm 1) ───────────────────────────
BUFFER_DISTANCE = 50.0  # metres

buffers = []
for seg in seg_objects:
    buf = build_segment_buffer(seg, distance_m=BUFFER_DISTANCE)
    buffers.append(buf)

print(f'Built {len(buffers)} buffer polygons ({BUFFER_DISTANCE}m distance)')

# ── Demonstrate one buffer ────────────────────────────────────────
demo_idx = 0
demo_seg = seg_objects[demo_idx]
demo_buf = buffers[demo_idx]
print(f'\nDemo: {seg_names[demo_idx]}')
print(f'  Segment: ({demo_seg.p1.x:.6f}, {demo_seg.p1.y:.6f}) -> '
      f'({demo_seg.p2.x:.6f}, {demo_seg.p2.y:.6f})')
print(f'  Length: {demo_seg.length():.1f} m')
print(f'  Buffer vertices:')
for i, v in enumerate(demo_buf.vertices):
    print(f'    V{i}: ({v.x:.6f}, {v.y:.6f})')
centroid = demo_buf.calculate_centroid()
print(f'  Buffer centroid: ({centroid.x:.6f}, {centroid.y:.6f})')
print(f'  Buffer area: {demo_buf.calculate_area():.10f} sq degrees')

# ── Demonstrate segment intersection (Lecture 3 – turn test) ──────
if len(seg_objects) >= 2:
    cross = segments_intersect(seg_objects[0], seg_objects[1])
    print(f'\nSegments 0 and 1 intersect: {cross}')

Created 5559 Segment objects
Built 5559 buffer polygons (50.0m distance)

Demo: Häringstrasse
  Segment: (8.544509, 47.376738) -> (8.547257, 47.373004)
  Length: 463.8 m
  Buffer vertices:
    V0: (8.545043, 47.377004)
    V1: (8.547791, 47.373271)
    V2: (8.546723, 47.372738)
    V3: (8.543974, 47.376471)
  Buffer centroid: (8.545883, 47.374871)
  Buffer area: 0.0000054517 sq degrees

Segments 0 and 1 intersect: False


# 8. Point-in-Polygon: Spatial Join & Sound Profile Computation

**Algorithm 2 (Lecture 4):** Ray-casting point-in-polygon with even-odd rule (Jordan Curve Theorem).
1. Bounding-box pre-filter (O(1)) rejects most non-candidate points.
2. Cast a horizontal ray from the test point to +∞.
3. Count intersections with polygon edges.
4. Odd count → point is inside.

**Algorithm 3:** Sound profile computation with z-score normalisation (Aiello et al., 2016).
- `sound(j,c) = tags(j,c) / tags(j)` — fraction per category per segment.
- `z(j,c) = (sound(j,c) - μ_c) / σ_c` — z-score normalisation for cross-category comparison.
- Dominant sound = category with highest z-score.

In [10]:
from geo_proxy.pipeline import assign_sound_category, spatial_join, CATEGORY_WORD_SETS
from geo_proxy.algorithms import compute_sound_profile, zscore_normalise, dominant_sound
from geo_proxy.validation import parse_csv_points, parse_csv_noise, run_validation
from collections import Counter

# ── Load sound observation data ──────────────────────────────────
sound_points = parse_csv_points('data/zurich_sounds.csv')
print(f'Loaded {len(sound_points)} sound observation points')

# Show category distribution
cat_dist = Counter(p['sound_category'] for p in sound_points)
print('Category distribution:')
for cat, cnt in sorted(cat_dist.items(), key=lambda x: -x[1]):
    print(f'  {cat:12s}: {cnt:3d}')

# ── Run full spatial join pipeline (Algorithms 1+2+3) ────────────
print('\nRunning spatial join pipeline ...')
results = spatial_join(seg_objects, sound_points, buffer_distance=BUFFER_DISTANCE)
print(f'Processed {len(results)} street segments')

# ── Results summary ──────────────────────────────────────────────
sound_dist = Counter(r['dominant_sound'] for r in results)
print(f'\nDominant sound distribution:')
for cat, count in sorted(sound_dist.items(), key=lambda x: -x[1]):
    pct = count / len(results) * 100
    print(f'  {cat:12s}: {count:4d} segments ({pct:.1f}%)')

matched = [r for r in results if r['matched_points'] > 0]
print(f'\nSegments with >= 1 matched point: {len(matched)} / {len(results)}')
if matched:
    avg = sum(r['matched_points'] for r in matched) / len(matched)
    print(f'Average matched points per segment: {avg:.1f}')

# ── Validation against noise data ────────────────────────────────
noise_data = parse_csv_noise('data/zurich_noise.csv')
print(f'\nLoaded {len(noise_data)} noise measurement points')

val = run_validation(results, noise_data)
print(f'Spearman rank correlation (transport z-score vs dB):')
print(f'  rho = {val["spearman_rho"]:.4f}')
print(f'  n   = {val["n_matched_segments"]} matched segments')
print(f'  {val["interpretation"]}')

Loaded 800 sound observation points
Category distribution:
  transport   : 185
  nature      : 179
  human       : 167
  music       : 107
  indoor      :  98
  mechanical  :  64

Running spatial join pipeline ...
Processed 5559 street segments

Dominant sound distribution:
  none        : 5225 segments (94.0%)
  nature      :   69 segments (1.2%)
  human       :   64 segments (1.2%)
  music       :   53 segments (1.0%)
  indoor      :   53 segments (1.0%)
  mechanical  :   48 segments (0.9%)
  transport   :   47 segments (0.8%)

Segments with >= 1 matched point: 334 / 5559
Average matched points per segment: 5.4

Loaded 150 noise measurement points
Spearman rank correlation (transport z-score vs dB):
  rho = 0.1444
  n   = 496 matched segments
  Positive correlation supports the Chatty Maps hypothesis.


9. Visualization

In [11]:
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for Colab compatibility
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

COLOURS = {
    'transport':  '#e74c3c',
    'nature':     '#2ecc71',
    'human':      '#3498db',
    'music':      '#9b59b6',
    'mechanical': '#e67e22',
    'indoor':     '#95a5a6',
    'none':       '#ecf0f1',
}

# ── Plot 1: Sound map ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 10))
for r in results:
    seg = r['segment']
    colour = COLOURS.get(r['dominant_sound'], '#cccccc')
    lw = 2.5 if r['matched_points'] > 0 else 0.4
    alpha = 0.9 if r['matched_points'] > 0 else 0.2
    ax.plot([seg.p1.x, seg.p2.x], [seg.p1.y, seg.p2.y],
            color=colour, linewidth=lw, alpha=alpha)

patches = [mpatches.Patch(color=c, label=cat)
           for cat, c in COLOURS.items() if cat != 'none']
ax.legend(handles=patches, loc='upper left', fontsize=9)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Chatty Maps Zürich – Dominant Sound per Street Segment')
plt.tight_layout()
plt.savefig('output_soundmap.png', dpi=150)
plt.show()
print('Sound map saved to output_soundmap.png')

# ── Plot 2: Buffer example ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
example = next((r for r in results if r['matched_points'] >= 2), results[0])
seg = example['segment']
buf = example['buffer']

verts = [(v.x, v.y) for v in buf.vertices] + [(buf.vertices[0].x, buf.vertices[0].y)]
xs, ys = zip(*verts)
ax.fill(xs, ys, alpha=0.2, color='blue', label=f'{BUFFER_DISTANCE}m buffer')
ax.plot(xs, ys, 'b--', linewidth=1)
ax.plot([seg.p1.x, seg.p2.x], [seg.p1.y, seg.p2.y],
        'b-', linewidth=3, label='Street segment')

for p in sound_points:
    if point_in_polygon(p['geometry'], buf):
        c = COLOURS.get(p['sound_category'], 'gray')
        ax.plot(p['geometry'].x, p['geometry'].y, 'o', color=c, markersize=8)

c = example['centroid']
ax.plot(c.x, c.y, 'kx', markersize=12, markeredgewidth=2, label='Centroid')
ax.legend(fontsize=8)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title(f"Buffer Example — Dominant: {example['dominant_sound']} "
             f"({example['matched_points']} points)")
plt.tight_layout()
plt.savefig('output_buffer_example.png', dpi=150)
plt.show()
print('Buffer example saved to output_buffer_example.png')

C:\Users\gfilg\AppData\Local\Temp\ipykernel_2072\1930368377.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Sound map saved to output_soundmap.png
Buffer example saved to output_buffer_example.png


C:\Users\gfilg\AppData\Local\Temp\ipykernel_2072\1930368377.py:64: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
